# Part 5: Build an agent over the knowledge base

Part 5 at Caldova begins when the supply-chain team wants an agent to reason over evidence without asking the knowledge base to synthesize an answer. You will create an extractive knowledge base from the Part 1 knowledge sources, connect Microsoft Agent Framework to its MCP endpoint, then let the agent work with the retrieved evidence.

**🎯 Mission**
- Create a model-free knowledge base configured for extractive output
- Connect Microsoft Agent Framework to the knowledge base through its MCP endpoint
- Run an operational query and inspect the agent's tool activity

## Step 1: Set up the agent environment

Load the Azure OpenAI model and Foundry IQ settings from the project `.env` file. These settings create the extractive knowledge base and connect it to the Agent Framework model.

In [ ]:
import os

from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework_openai import OpenAIChatClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import AsyncOpenAI

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
MODEL_DEPLOYMENT_NAME = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]

KNOWLEDGE_BASE_NAME = "caldova-supply-chain-minimal-knowledge-base"
AZURE_SEARCH_API_VERSION = "2026-08-01-preview"

print("Agent environment loaded")

## Step 2: Add authentication to MCP requests

The knowledge base MCP endpoint accepts the Azure AI Search admin key that is preconfigured for this lab. Define a header provider that adds the key to each MCP request.

In [ ]:
def get_mcp_headers(_):
    return {"api-key": AZURE_SEARCH_ADMIN_KEY}

## Step 3: Create an extractive knowledge base

Create a new knowledge base over the three knowledge sources from Part 1. Minimal reasoning effort does not perform source selection, query planning, or iterative search. `EXTRACTIVE_DATA` returns retrieved evidence without attaching a model or synthesizing a knowledge base answer.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

knowledge_source_names = [
    "file-docs-knowledge-source",
    "supplier-evidence-knowledge-source",
    "sourcing-documents-knowledge-source",
]

index_client = SearchIndexClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    credential=AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY),
)
knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Caldova extractive knowledge base for supply-chain evidence",
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in knowledge_source_names],
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Extractive knowledge base '{KNOWLEDGE_BASE_NAME}' created or updated")

## Step 4: Create the knowledge base MCP tool

Every Foundry IQ knowledge base exposes a streamable HTTP MCP endpoint. Build the extractive knowledge base endpoint and expose only the `knowledge_base_retrieve` operation to the agent.

In [ ]:
knowledge_base_endpoint = (
    f"{AZURE_SEARCH_SERVICE_ENDPOINT.rstrip('/')}"
    f"/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp"
    f"?api-version={AZURE_SEARCH_API_VERSION}"
)
knowledge_base_mcp_tool = MCPStreamableHTTPTool(
    name="caldova-knowledge-base",
    url=knowledge_base_endpoint,
    header_provider=get_mcp_headers,
    allowed_tools=["knowledge_base_retrieve"],
    load_prompts=False,
)

print(f"MCP tool configured for {knowledge_base_endpoint}")

## Step 5: Create the supply-chain agent

Create a code-first Agent Framework agent backed by the Azure OpenAI model deployed for the lab. The knowledge base returns extracts without synthesis, while the agent reasons over that evidence to produce the final answer.

In [ ]:
openai_client = AsyncOpenAI(
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=AZURE_OPENAI_KEY,
)
chat_client = OpenAIChatClient(
    model=MODEL_DEPLOYMENT_NAME,
    async_client=openai_client,
)
agent = Agent(
    client=chat_client,
    name="CaldovaSupplyChainAgent",
    instructions=(
        "You are a Caldova supply-chain assistant. "
        "Always use the Caldova knowledge base tool for supply-chain questions. "
        "Ground every answer in retrieved evidence, cite the source titles or identifiers, "
        "and keep policy, approved commitments, and operational evidence distinct. "
        "If the retrieved evidence is insufficient, say what information is unavailable."
    ),
    tools=[knowledge_base_mcp_tool],
    default_options={"store": False},
)

print(f"Agent '{agent.name}' created")

## Step 6: Run a query

Open the MCP connection for one query, then close it automatically when the response completes. Change `question` and rerun this cell to ask additional supply-chain questions without leaving an MCP lifecycle task running.

In [ ]:
question = (
    "Shipment SHIP-VC-1026-BOS-FRA exceeded its temperature range. "
    "What happened, and can the 4,800 units be distributed?"
)

async with knowledge_base_mcp_tool:
    response = await agent.run(question)

display(Markdown(response.text))

## Step 7: Inspect agent activity

Agent responses include the messages produced during `run()`. Display each MCP tool call and its arguments to see the query the agent sent to the knowledge base before producing its final answer.

In [ ]:
import json

table_rows = []
for message in response.messages:
    for content in message.contents:
        if content.type == "function_call":
            arguments = content.arguments
            if not isinstance(arguments, str):
                arguments = json.dumps(arguments, indent=2)
            arguments = arguments.replace("|", "\\|").replace("\n", "<br>")
            table_rows.append((content.name, arguments))

table = ["| Call | Tool | Arguments |", "| ---: | --- | --- |"]
for call_number, (tool_name, arguments) in enumerate(table_rows, start=1):
    table.append(f"| {call_number} | `{tool_name}` | `{arguments}` |")

display(Markdown("\n".join(table)))

## ✅ Mission complete

**What you built:**

* ✓ **Extractive knowledge base**: A model-free knowledge base returns evidence from the Part 1 sources without answer synthesis.
* ✓ **Authenticated MCP connection**: The agent reaches the extractive knowledge base with the lab's configured Search key.
* ✓ **Agent Framework supply-chain agent**: An Azure OpenAI-backed agent reasons over the retrieved evidence to answer an operational question.

➡️ Continue to the [lab summary](../../instructions/summary.md).